# Random Forest Classification: Six Optimization Algorithms Comparison

**Course Project | Hyperparameter Optimization with NumPy + Scikit-learn**

This notebook compares six optimization algorithms for tuning a Random Forest binary-classification model. The experiment uses the Breast Cancer Wisconsin Diagnostic dataset and evaluates the final models using Accuracy, Precision, Recall, F1-score, and Log Loss.

**Important methodological note:** Random Forest is a tree-ensemble algorithm and is not trained with gradient descent. Therefore, the six optimizers are used to search for good Random Forest hyperparameters. Since the validation-loss objective is a black box and is not differentiable, we estimate its search direction using **SPSA (Simultaneous Perturbation Stochastic Approximation)**. The optimizer update rules (GD, SGD, Mini-Batch GD, Momentum, RMSprop, and Adam) are then applied to the hyperparameter vector—not to the internal tree training process.


## 1. Dataset and experiment setup

We use the Breast Cancer Wisconsin Diagnostic dataset available directly through `scikit-learn`, so no external Kaggle upload is required.

- **Task:** Binary classification (malignant vs. benign)
- **Input:** 30 numerical features
- **Target:** diagnosis class
- **Split:** 60% training, 20% validation, 20% testing (stratified)
- **Random state:** 42

The validation set is used for hyperparameter optimization. The test set is kept separate and is used only for the final evaluation.


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, roc_auc_score
)

warnings.filterwarnings("ignore")
np.random.seed(42)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## 2. Load the dataset

The original dataset target is encoded as `0 = malignant` and `1 = benign` in scikit-learn. We keep this binary encoding and also display the class names for clarity.


In [ ]:
dataset = load_breast_cancer()
X = dataset.data.astype(float)
y_original = dataset.target.astype(int)

# Convert target so that 1 = malignant and 0 = benign.
# This makes malignant the positive class for Precision, Recall, and F1.
y = (y_original == 0).astype(int)

feature_names = list(dataset.feature_names)
target_names = ["Benign (0)", "Malignant (1)"]

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

print("Dataset shape:", df.shape)
print("Number of features:", X.shape[1])
print("Class distribution:")
print(df["target"].value_counts().rename(index={0: "Benign (0)", 1: "Malignant (1)"}))
display(df.head())


## 3. Exploratory data analysis

We inspect missing values, class balance, descriptive statistics, and a small selection of feature distributions.


In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
display(df.describe().T.head(10))

class_counts = df["target"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["Benign (0)", "Malignant (1)"], class_counts.values)
plt.title("Class Distribution")
plt.ylabel("Number of samples")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

selected_features = ["mean radius", "mean texture", "mean perimeter", "mean area"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.ravel(), selected_features):
    ax.hist(X[y == 0, feature_names.index(feature)], bins=20, alpha=0.65, label="Benign")
    ax.hist(X[y == 1, feature_names.index(feature)], bins=20, alpha=0.65, label="Malignant")
    ax.set_title(feature)
    ax.legend()
plt.tight_layout()
plt.show()


## 4. Train/validation/test split

We first reserve 20% of the data for the final test. The remaining 80% is split into training and validation sets. Stratification preserves the class proportions in all splits.

Random Forest does not require feature scaling, so the original numerical feature values are used.


In [ ]:
RANDOM_STATE = 42
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25,
    random_state=RANDOM_STATE, stratify=y_train_full
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)
print("Training positive rate:", round(float(y_train.mean()), 4))
print("Validation positive rate:", round(float(y_val.mean()), 4))
print("Test positive rate:", round(float(y_test.mean()), 4))


## 5. Random Forest model and hyperparameter search space

A Random Forest combines multiple decision trees. Each tree is trained using a bootstrap sample and considers a subset of features when splitting.

We optimize five hyperparameters:

| Hyperparameter | Search range | Meaning |
|---|---:|---|
| `n_estimators` | 40–160 | Number of trees |
| `max_depth` | 2–20 | Maximum tree depth |
| `min_samples_split` | 2–20 | Minimum samples required to split |
| `min_samples_leaf` | 1–10 | Minimum samples in a leaf |
| `max_features` | 0.30–1.00 | Fraction of features considered per split |

The optimizer works on a normalized vector between 0 and 1. A decoding function converts that vector into valid Random Forest settings.


In [ ]:
PARAMETER_NAMES = [
    "n_estimators", "max_depth", "min_samples_split",
    "min_samples_leaf", "max_features"
]

LOWER = np.array([40, 2, 2, 1, 0.30], dtype=float)
UPPER = np.array([160, 20, 20, 10, 1.00], dtype=float)

def decode_hyperparameters(vector):
    vector = np.clip(np.asarray(vector, dtype=float), 0.0, 1.0)
    values = LOWER + vector * (UPPER - LOWER)
    return {
        "n_estimators": int(np.clip(round(values[0] / 10) * 10, 40, 160)),
        "max_depth": int(np.clip(round(values[1]), 2, 20)),
        "min_samples_split": int(np.clip(round(values[2]), 2, 20)),
        "min_samples_leaf": int(np.clip(round(values[3]), 1, 10)),
        "max_features": float(np.clip(values[4], 0.30, 1.00))
    }

initial_vector = np.array([0.50, 0.50, 0.25, 0.20, 0.70])
print("Initial hyperparameters:")
print(decode_hyperparameters(initial_vector))


## 6. Validation loss (objective function)

The optimization objective is **validation Log Loss**. Lower Log Loss indicates better calibrated probability predictions.

The model is fitted on the training data (or a training subset for SGD and Mini-Batch GD) and evaluated on the validation data. The test set is not accessed during optimization.

SPSA estimates a gradient using two objective evaluations per iteration:

\[
\hat{g}_i = \frac{L(\theta + c\Delta)-L(\theta-c\Delta)}{2c\Delta_i}
\]

where each component of \(\Delta\) is randomly either -1 or +1.


In [ ]:
def objective(vector, method="GD", iteration=0):
    params = decode_hyperparameters(vector)

    # Full data for GD-style methods; stochastic subsets for SGD / Mini-Batch.
    rng = np.random.default_rng(RANDOM_STATE + iteration)
    if method == "SGD":
        sample_size = max(80, int(0.35 * len(X_train)))
        idx = rng.choice(len(X_train), size=min(sample_size, len(X_train)), replace=False)
        X_fit, y_fit = X_train[idx], y_train[idx]
    elif method == "Mini-Batch GD":
        sample_size = max(120, int(0.65 * len(X_train)))
        idx = rng.choice(len(X_train), size=min(sample_size, len(X_train)), replace=False)
        X_fit, y_fit = X_train[idx], y_train[idx]
    else:
        X_fit, y_fit = X_train, y_train

    model = RandomForestClassifier(
        **params, random_state=RANDOM_STATE, n_jobs=-1,
        class_weight=None
    )
    model.fit(X_fit, y_fit)
    probabilities = model.predict_proba(X_val)[:, 1]
    return float(log_loss(y_val, probabilities, labels=[0, 1]))

print("Initial validation Log Loss:", round(objective(initial_vector), 5))


## 7. Implement six optimizers

The six methods mirror the first assignment:

- **GD:** full-data SPSA direction and a standard update.
- **SGD:** SPSA objective estimates use a random training subset.
- **Mini-Batch GD:** estimates use a larger random training subset.
- **Momentum:** adds a velocity term to smooth updates.
- **RMSprop:** scales updates using a moving average of squared gradients.
- **Adam:** combines momentum and RMSprop with bias correction.

Because the objective is non-differentiable, these are derivative-free/SPSA-based adaptations of the optimizer update rules. They are not gradients through the Random Forest trees.


In [ ]:
def optimize_hyperparameters(
    method, epochs=18, learning_rate=0.08, c=0.08,
    beta=0.9, beta2=0.999, epsilon=1e-8
):
    rng = np.random.default_rng(RANDOM_STATE)
    theta = initial_vector.astype(float).copy()
    velocity = np.zeros_like(theta)
    squared_avg = np.zeros_like(theta)
    first_moment = np.zeros_like(theta)
    second_moment = np.zeros_like(theta)
    history = []
    best_theta = theta.copy()
    best_loss = np.inf

    for epoch in range(epochs):
        delta = rng.choice([-1.0, 1.0], size=len(theta))
        loss_plus = objective(theta + c * delta, method, epoch)
        loss_minus = objective(theta - c * delta, method, epoch)
        gradient_estimate = ((loss_plus - loss_minus) / (2.0 * c)) * delta

        # Clip unusually large noisy estimates for stable search.
        grad_norm = np.linalg.norm(gradient_estimate)
        if grad_norm > 5.0:
            gradient_estimate = gradient_estimate * (5.0 / grad_norm)

        timestep = epoch + 1
        if method in ["GD", "SGD", "Mini-Batch GD"]:
            theta -= learning_rate * gradient_estimate
        elif method == "Momentum":
            velocity = beta * velocity + learning_rate * gradient_estimate
            theta -= velocity
        elif method == "RMSprop":
            squared_avg = beta * squared_avg + (1 - beta) * (gradient_estimate ** 2)
            theta -= learning_rate * gradient_estimate / (np.sqrt(squared_avg) + epsilon)
        elif method == "Adam":
            first_moment = beta * first_moment + (1 - beta) * gradient_estimate
            second_moment = beta2 * second_moment + (1 - beta2) * (gradient_estimate ** 2)
            m_hat = first_moment / (1 - beta ** timestep)
            v_hat = second_moment / (1 - beta2 ** timestep)
            theta -= learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
        else:
            raise ValueError(f"Unknown optimizer: {method}")

        theta = np.clip(theta, 0.0, 1.0)
        current_loss = objective(theta, method, epoch)
        history.append(current_loss)

        if current_loss < best_loss:
            best_loss = current_loss
            best_theta = theta.copy()

    return best_theta, np.asarray(history), best_loss

OPTIMIZERS = ["GD", "SGD", "Mini-Batch GD", "Momentum", "RMSprop", "Adam"]
print("Optimizers:", OPTIMIZERS)


## 8. Run all six optimization experiments

Each optimizer searches for a hyperparameter configuration by minimizing validation Log Loss. We record the best validation loss, runtime, and selected hyperparameters.

The number of iterations is kept moderate so that the notebook can run in Kaggle or Google Colab without an excessively long execution time.


In [ ]:
EPOCHS = 18
LEARNING_RATE = 0.08

optimization_results = []
best_parameters = {}
loss_histories = {}

for optimizer_name in OPTIMIZERS:
    print(f"Running {optimizer_name}...")
    start_time = time.perf_counter()

    best_vector, history, best_val_loss = optimize_hyperparameters(
        optimizer_name, epochs=EPOCHS, learning_rate=LEARNING_RATE
    )
    elapsed = time.perf_counter() - start_time
    params = decode_hyperparameters(best_vector)

    best_parameters[optimizer_name] = params
    loss_histories[optimizer_name] = history
    optimization_results.append({
        "Optimizer": optimizer_name,
        "Best Validation Log Loss": best_val_loss,
        "Training/Optimization Time (s)": elapsed,
        **params
    })

optimization_df = pd.DataFrame(optimization_results)
display(optimization_df.sort_values("Best Validation Log Loss").round(5))


## 9. Compare optimization convergence

The plot shows the validation Log Loss recorded after each hyperparameter update. The curve is a black-box optimization trace, not the internal training loss of individual trees.


In [ ]:
plt.figure(figsize=(11, 6))
for optimizer_name in OPTIMIZERS:
    plt.plot(loss_histories[optimizer_name], marker="o", linewidth=1.7, label=optimizer_name)
plt.title("Validation Log Loss — Optimizer Comparison")
plt.xlabel("Optimization Iteration")
plt.ylabel("Validation Log Loss (lower is better)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 10. Train final Random Forest models

For each optimizer, we fit a new Random Forest using its best hyperparameters and all development data (training + validation). The held-out test set remains untouched until this final evaluation.


In [ ]:
X_development = np.vstack([X_train, X_val])
y_development = np.concatenate([y_train, y_val])

final_models = {}
test_probabilities = {}
test_predictions = {}
final_results = []

for optimizer_name in OPTIMIZERS:
    params = best_parameters[optimizer_name]
    model = RandomForestClassifier(
        **params, random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X_development, y_development)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    final_models[optimizer_name] = model
    test_probabilities[optimizer_name] = probabilities
    test_predictions[optimizer_name] = predictions

    final_results.append({
        "Optimizer": optimizer_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-Score": f1_score(y_test, predictions, zero_division=0),
        "Log Loss": log_loss(y_test, probabilities, labels=[0, 1])
    })

results_df = pd.DataFrame(final_results)
display(results_df.sort_values("F1-Score", ascending=False).round(5))


## 11. Evaluation metrics

We report five classification metrics:

- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** fraction of predicted malignant cases that are truly malignant.
- **Recall:** fraction of all malignant cases correctly detected.
- **F1-Score:** harmonic mean of Precision and Recall.
- **Log Loss:** evaluates the quality of predicted probabilities; lower is better.

Malignant is treated as the positive class (`1`).


In [ ]:
metric_columns = ["Accuracy", "Precision", "Recall", "F1-Score"]
display(results_df.set_index("Optimizer")[metric_columns + ["Log Loss"]].round(4))

ax = results_df.set_index("Optimizer")[metric_columns].plot(
    kind="bar", figsize=(12, 6), ylim=(0, 1.05)
)
ax.set_title("Classification Metrics by Optimizer")
ax.set_ylabel("Score")
ax.set_xlabel("Optimizer")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["Optimizer"], results_df["Log Loss"])
plt.title("Test Log Loss by Optimizer (lower is better)")
plt.ylabel("Log Loss")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 12. Confusion matrices

A confusion matrix shows true negatives, false positives, false negatives, and true positives. False negatives are particularly important in this medical screening example because they represent malignant cases predicted as benign.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, optimizer_name in zip(axes.ravel(), OPTIMIZERS):
    cm = confusion_matrix(y_test, test_predictions[optimizer_name], labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=["Benign", "Malignant"]
    )
    disp.plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(optimizer_name)
plt.tight_layout()
plt.show()


## 13. ROC curves and AUC

ROC curves compare the true-positive rate with the false-positive rate across classification thresholds. AUC summarizes the ranking performance; higher is better.


In [ ]:
plt.figure(figsize=(9, 7))
for optimizer_name in OPTIMIZERS:
    probabilities = test_probabilities[optimizer_name]
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc_value = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, linewidth=1.8, label=f"{optimizer_name} (AUC={auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 14. Overall optimizer comparison

We rank optimizers by test F1-score (higher is better) and also report Log Loss (lower is better). These rankings are specific to this dataset and this experiment; they do not prove that one optimizer is universally best.


In [ ]:
ranked_results = results_df.copy()
ranked_results["F1 Rank"] = ranked_results["F1-Score"].rank(
    method="min", ascending=False
)
ranked_results["Log Loss Rank"] = ranked_results["Log Loss"].rank(
    method="min", ascending=True
)
display(
    ranked_results.sort_values("F1 Rank")[
        ["Optimizer", "F1-Score", "F1 Rank", "Log Loss", "Log Loss Rank", "Accuracy", "Precision", "Recall"]
    ].round(5)
)

plt.figure(figsize=(10, 5))
ranked_results.sort_values("F1-Score").plot(
    x="Optimizer", y="F1-Score", kind="bar", legend=False, figsize=(10, 5)
)
plt.title("F1-Score Ranking (higher is better)")
plt.ylabel("F1-Score")
plt.xlabel("Optimizer")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 15. Best optimizer and selected hyperparameters

The best optimizer is selected from the measured test F1-score. We also display the corresponding hyperparameters. For a formal scientific report, validation performance should be used for model selection and the test set should be treated as a final unbiased estimate; therefore, do not repeatedly tune the model based on the test results.


In [ ]:
best_optimizer = results_df.loc[results_df["F1-Score"].idxmax(), "Optimizer"]
print("Highest test F1-score in this run:", best_optimizer)
print("Selected hyperparameters:")
for key, value in best_parameters[best_optimizer].items():
    print(f"  {key}: {value}")

print("\nClassification report for the selected model:")
print(classification_report(
    y_test,
    test_predictions[best_optimizer],
    target_names=["Benign", "Malignant"],
    zero_division=0
))


## 16. Discussion and limitations

1. Random Forest is an ensemble of decision trees. It does not use gradient descent to learn tree splits.
2. The six optimizers in this experiment tune the Random Forest hyperparameters using an SPSA-based estimate of the black-box validation-loss direction.
3. SGD and Mini-Batch variants use random subsets of the training data when estimating the objective. This makes their search noisier and computationally different from full-data methods.
4. Hyperparameter optimization is stochastic. Results may change with the random seed, number of iterations, and search-space boundaries.
5. The Breast Cancer Wisconsin dataset is a relatively small benchmark dataset. Performance on it does not guarantee performance on other medical datasets.
6. The test set is reserved for final evaluation. For stronger research, use repeated stratified cross-validation on the development data and evaluate once on a separate test set.
7. The best optimizer in this experiment is not necessarily the best optimizer for every Random Forest dataset or search space.

### Final conclusion

This notebook compares six optimization algorithms—GD, SGD, Mini-Batch GD, Momentum, RMSprop, and Adam—as hyperparameter-search strategies for Random Forest binary classification. SPSA is used to estimate search directions because the validation Log Loss is a black-box, non-differentiable objective. The final models are compared using Accuracy, Precision, Recall, F1-Score, Log Loss, confusion matrices, and ROC-AUC. The results generated by running the notebook should be used in the final report rather than claiming a predetermined winner.


## 17. Export results

The following files are saved to the current working directory. In Kaggle, they will be available in the notebook Output panel after execution.


In [ ]:
results_df.to_csv("random_forest_optimizer_results.csv", index=False)
optimization_df.to_csv("random_forest_optimization_details.csv", index=False)

hyperparameter_rows = []
for optimizer_name, params in best_parameters.items():
    hyperparameter_rows.append({"Optimizer": optimizer_name, **params})
hyperparameters_df = pd.DataFrame(hyperparameter_rows)
hyperparameters_df.to_csv("random_forest_best_hyperparameters.csv", index=False)

print("Saved:")
print("- random_forest_optimizer_results.csv")
print("- random_forest_optimization_details.csv")
print("- random_forest_best_hyperparameters.csv")


## 18. References

- Scikit-learn: Breast Cancer Wisconsin Diagnostic dataset — https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html
- Scikit-learn: RandomForestClassifier — https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
- Scikit-learn: Classification metrics — https://scikit-learn.org/stable/modules/model_evaluation.html
- Spall, J. C. (1998). *An Overview of the Simultaneous Perturbation Method for Efficient Optimization*. Johns Hopkins APL Technical Digest.

**Reproducibility:** random seed = 42. Libraries: Python, NumPy, Pandas, Matplotlib, and Scikit-learn.
